# 02 - yfinance Scraping

This notebook collects market data from yfinance/Yahoo Finance.

Final CSV outputs:
- `data/yfinance_stock_prices.csv`
- `data/yfinance_company_profile.csv`
- `data/yfinance_dividends.csv`

In [ ]:
!pip -q install pandas yfinance curl_cffi

In [ ]:
from pathlib import Path
import pandas as pd
import yfinance as yf
from curl_cffi import requests as curl_requests
try:
    from IPython.display import display
except ImportError:
    # Local validation fallback. Colab has display(), but plain Python may not.
    def display(value):
        print(value)

DATA_DIR = Path("/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = "2021-01-01"
TICKERS = {
    "SRU-UN.TO": "SmartCentres REIT",
    "REI-UN.TO": "RioCan REIT",
    "CHP-UN.TO": "Choice Properties REIT",
    "AP-UN.TO": "Allied Properties REIT",
    "FCR-UN.TO": "First Capital REIT",
}
yf_session = curl_requests.Session(impersonate="chrome")
# Local Windows certificate stores can fail with Yahoo requests. Colab usually
# works normally, but this keeps the notebook robust on this PC too.
yf_session.verify = False

In [ ]:
# Daily prices support later return, volatility, and peer performance analysis.
# Adjusted Close is usually the most useful price column for return analysis.
stock_frames = []
for ticker, company_name in TICKERS.items():
    history = yf.Ticker(ticker, session=yf_session).history(start=START_DATE, auto_adjust=False)
    if history.empty:
        print("No stock history returned for", ticker)
        continue
    history = history.reset_index()
    keep_columns = [column for column in ["Date", "Adj Close", "Close", "High", "Low", "Open", "Volume"] if column in history.columns]
    history = history[keep_columns]
    history.insert(0, "company_name", company_name)
    history.insert(0, "ticker", ticker)
    stock_frames.append(history)

stock_prices_df = pd.concat(stock_frames, ignore_index=True) if stock_frames else pd.DataFrame()
stock_prices_df.to_csv(DATA_DIR / "yfinance_stock_prices.csv", index=False)
print("Saved stock prices:", len(stock_prices_df))
display(stock_prices_df.head())

In [ ]:
# Company profile is a point-in-time peer snapshot.
# Market cap compares size; beta compares market sensitivity; dividend yield compares income.
fields = ["symbol", "shortName", "longName", "marketCap", "beta", "dividendYield", "sharesOutstanding", "trailingPE", "forwardPE", "fiftyTwoWeekHigh", "fiftyTwoWeekLow", "sector", "industry"]
profile_rows = []
for ticker in TICKERS:
    info = yf.Ticker(ticker, session=yf_session).info
    row = {"ticker": ticker}
    for field in fields:
        row[field] = info.get(field)
    profile_rows.append(row)

company_profile_df = pd.DataFrame(profile_rows)
company_profile_df.to_csv(DATA_DIR / "yfinance_company_profile.csv", index=False)
print("Saved profiles:", len(company_profile_df))
display(company_profile_df)

In [ ]:
# yfinance gives a longer SmartCentres distribution history than the visible company website table.
dividends = yf.Ticker("SRU-UN.TO", session=yf_session).dividends
if dividends.empty:
    yfinance_dividends_df = pd.DataFrame(columns=["ticker", "source", "dividend_date", "dividend_amount", "notes"])
else:
    dividends = dividends[dividends.index >= START_DATE]
    yfinance_dividends_df = dividends.reset_index()
    yfinance_dividends_df.columns = ["dividend_date", "dividend_amount"]
    yfinance_dividends_df.insert(0, "source", "yfinance")
    yfinance_dividends_df.insert(0, "ticker", "SRU-UN.TO")
    yfinance_dividends_df["notes"] = "Longer SmartCentres dividend/distribution history from Yahoo Finance"

yfinance_dividends_df.to_csv(DATA_DIR / "yfinance_dividends.csv", index=False)
print("Saved dividends:", len(yfinance_dividends_df))
display(yfinance_dividends_df.head())